# Experiment 1 - Competing Drives

## Scenario

In this experiment we will simulate an agent which has access to food and shelter.

When not at food, it senses its **hunger** increasing with each action. When at food, it resets to zero.

Shelter is the same - when away, the agent senses its **risk** increasing with each action. When at shelter, it resets to zero.

The agent equally dislikes hunger and risk. It increases exponentially, so doesn't mind a little but really hates a lot (as it becomes 'life-endangering').

These competing drives cause it to run back and forth between the shelter and food as it becomes overwhelmed by its desire to eat or hide.


## Rewards

This simulation demonstrates purely **extrinsic** value driven behaviour - the agent sees a state within the policy horizon that provides closer alignment with its preferred state (C matrix).

The agent believes it has complete information about the state (it's A matrix is identity, observations map 1:1 with hidden states). 

This means it doesn't have an incentive to seek observations that would make it more certain about its true state (**epistemic** value) in order to help it reach the goal.


## Policy Length

In this simulation, it is the (mandatory, singular) Hunger action ('eat') taken *after* landing on the food that causes relief, not the one that lands you there.

Similarly it is the (mandatory, singular) Stress action ('hide') taken *after* landing on the shelter that causes relief, not the one that lands you there.

This means that the agent has to be able to imagine the state one timestep *after* landing on the relevant goal to anticipate a reward.

The minimum policy length must therefore be one more than the Manhatten distance (L1, shortest path without diagonal jumps).


> I tried setting a belief that acting to *land* on a goal would cause you to see relief as that would reduce the policy length by one. The agent expected to always observe 0 stress/hunger at the goal locations (A matrix). Similarly, I updated the drive state in the env *after* updating location (reward if you arrived). This seemed sensible but the problem is that the B matrix can only say e.g. 'Given my current hunger and location, if I eat, what will be my resulting hunger'. The answer would depend on whether the agent moved into the goal (reset) or away from the goal (increment). It seemed that the only way to be deterministic is to say 'you have to eat *at* the food location'.



In [2]:
%pip install inferactively-pymdp

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
from pymdp.agent import Agent
from pymdp import utils

# Hidden States

We need to define a state space that incoporates
- Our **location** on a grid (index in list of coordinates)
- Our **hunger** level
- Our **stress** level



In [14]:
grid_dims = [4, 4]
num_grid_points = int(np.prod(grid_dims))
shelter_location = (0, 0)
food_location = (3, 0)
max_drive = 10

print(f"Grid dimensions: {grid_dims}")
print(f"Food location: {food_location}")
print(f"Shelter location: {shelter_location}")
print(f"Maximum drive level: {max_drive}")

Grid dimensions: [4, 4]
Food location: (3, 0)
Shelter location: (0, 0)
Maximum drive level: 10


In [15]:
# Create a look-up table `loc_list` that maps linear indices to (y, x) coordinates
grid = np.arange(num_grid_points).reshape(grid_dims)
it = np.nditer(grid, flags=["multi_index"])
loc_list = []
while not it.finished:
    loc_list.append(it.multi_index)
    it.iternext()

manhatten_distance = abs(food_location[0]-shelter_location[0]) + abs(food_location[1]-shelter_location[1])

print(f"Required policy length: {manhatten_distance + 1} steps") # Without a 'reward trail', policies must be at least the manhattan distance from the goal to see any (extrinsic) reward signal.

Required policy length: 4 steps


In [16]:
hunger_levels = np.arange(max_drive + 1)  # 0 to max_drive, inclusive
stress_levels = np.arange(max_drive + 1)  # 0 to max_drive, inclusive
num_hunger = len(hunger_levels)
num_stress = len(stress_levels)
num_states = [num_grid_points, num_hunger, num_stress]

print(f"Factor 0 (Location): {num_grid_points} states (one per grid cell)")
print(f"Factor 1 (Hunger): {num_hunger} states (0-{max_drive})")
print(f"Factor 2 (Stress): {num_stress} states (0-{max_drive})")

Factor 0 (Location): 16 states (one per grid cell)
Factor 1 (Hunger): 11 states (0-10)
Factor 2 (Stress): 11 states (0-10)


In [26]:
food_loc_idx = loc_list.index(food_location)
shelter_loc_idx = loc_list.index(shelter_location)
print(f"Shelter {shelter_location} = cell {shelter_loc_idx}\nFood {food_location} = cell {food_loc_idx}")

Shelter (0, 0) = cell 0
Food (3, 0) = cell 12
